<a href="https://colab.research.google.com/github/subham-28/PyTorch/blob/main/LSTM_next_word_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [126]:
!pip install nltk

In [127]:
from torch._C import import_ir_module
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
import nltk
from nltk.tokenize import word_tokenize

In [128]:
document = """About the Program
What is the course fee for  Data Science Mentorship Program (DSMP 2023)
The course follows a monthly subscription model where you have to make monthly payments of Rs 799/month.
What is the total duration of the course?
The total duration of the course is 7 months. So the total course fee becomes 799*7 = Rs 5600(approx.)
What is the syllabus of the mentorship program?
We will be covering the following modules:
Python Fundamentals
Python libraries for Data Science
Data Analysis
SQL for Data Science
Maths for Machine Learning
ML Algorithms
Practical ML
MLOPs
Case studies
You can check the detailed syllabus here - https://learnwith.campusx.in/courses/CampusX-Data-Science-Mentorship-Program-637339afe4b0615a1bbed390
Will Deep Learning and NLP be a part of this program?
No, NLP and Deep Learning both are not a part of this program’s curriculum.
What if I miss a live session? Will I get a recording of the session?
Yes all our sessions are recorded, so even if you miss a session you can go back and watch the recording.
Where can I find the class schedule?
Checkout this google sheet to see month by month time table of the course - https://docs.google.com/spreadsheets/d/16OoTax_A6ORAeCg4emgexhqqPv3noQPYKU7RJ6ArOzk/edit?usp=sharing.
What is the time duration of all the live sessions?
Roughly, all the sessions last 2 hours.
What is the language spoken by the instructor during the sessions?
Hinglish
How will I be informed about the upcoming class?
You will get a mail from our side before every paid session once you become a paid user.
Can I do this course if I am from a non-tech background?
Yes, absolutely.
I am late, can I join the program in the middle?
Absolutely, you can join the program anytime.
If I join/pay in the middle, will I be able to see all the past lectures?
Yes, once you make the payment you will be able to see all the past content in your dashboard.
Where do I have to submit the task?
You don’t have to submit the task. We will provide you with the solutions, you have to self evaluate the task yourself.
Will we do case studies in the program?
Yes.
Where can we contact you?
You can mail us at nitish.campusx@gmail.com
Payment/Registration related questions
Where do we have to make our payments? Your YouTube channel or website?
You have to make all your monthly payments on our website. Here is the link for our website - https://learnwith.campusx.in/
Can we pay the entire amount of Rs 5600 all at once?
Unfortunately no, the program follows a monthly subscription model.
What is the validity of monthly subscription? Suppose if I pay on 15th Jan, then do I have to pay again on 1st Feb or 15th Feb
15th Feb. The validity period is 30 days from the day you make the payment. So essentially you can join anytime you don’t have to wait for a month to end.
What if I don’t like the course after making the payment. What is the refund policy?
You get a 7 days refund period from the day you have made the payment.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmail.com
Post registration queries
Till when can I view the paid videos on the website?
This one is tricky, so read carefully. You can watch the videos till your subscription is valid. Suppose you have purchased subscription on 21st Jan, you will be able to watch all the past paid sessions in the period of 21st Jan to 20th Feb. But after 21st Feb you will have to purchase the subscription again.
But once the course is over and you have paid us Rs 5600(or 7 installments of Rs 799) you will be able to watch the paid sessions till Aug 2024.
Why lifetime validity is not provided?
Because of the low course fee.
Where can I reach out in case of a doubt after the session?
You will have to fill a google form provided in your dashboard and our team will contact you for a 1 on 1 doubt clearance session
If I join the program late, can I still ask past week doubts?
Yes, just select past week doubt in the doubt clearance google form.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmai.com
Certificate and Placement Assistance related queries
What is the criteria to get the certificate?
There are 2 criterias:
You have to pay the entire fee of Rs 5600
You have to attempt all the course assessments.
I am joining late. How can I pay payment of the earlier months?
You will get a link to pay fee of earlier months in your dashboard once you pay for the current month.
I have read that Placement assistance is a part of this program. What comes under Placement assistance?
This is to clarify that Placement assistance does not mean Placement guarantee. So we dont guarantee you any jobs or for that matter even interview calls. So if you are planning to join this course just for placements, I am afraid you will be disappointed. Here is what comes under placement assistance
Portfolio Building sessions
Soft skill sessions
Sessions with industry mentors
Discussion on Job hunting strategies
"""

In [129]:
# tokenization
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [130]:
#tokenize

tokens=word_tokenize(document.lower())

In [131]:
#vocabs

vocab={'<unk>':0}
for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token]=len(vocab)



In [132]:
len(vocab)

289

In [133]:
#extract sentence from vocab
input_sentences=document.split('\n')

In [134]:
def text_to_indices(sentence, vocab):
  numerical_sentence=[]
  for token in sentence:
    if token not in vocab:
      numerical_sentence.append(vocab['<unk>'])
    else:
      numerical_sentence.append(vocab[token])
  return numerical_sentence

In [135]:
input_numerical_sentences=[]
for sentence in input_sentences:
  input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))


In [136]:
len(input_numerical_sentences)

78

In [137]:
training_sequence=[]
for sentence in input_numerical_sentences:
  for i in range(1,len(sentence)):
      training_sequence.append(sentence[:i+1])

In [138]:
len(training_sequence)

942

In [139]:
len_list=[]
for seq in training_sequence:
  len_list.append(len(seq))

In [140]:
max_len=max(len_list)

In [141]:
padded_sequences = []
for sentence in training_sequence:
  rem=max_len-len(sentence)
  padded_sequences.append([0]*rem + sentence)

In [142]:
len(padded_sequences[0])

62

In [143]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [144]:
padded_training_sequence=torch.tensor(padded_sequences,dtype=torch.long)
padded_training_sequence.to(device)

tensor([[  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        [  0,   0,   0,  ...,   0,   4,   5],
        ...,
        [  0,   0,   0,  ..., 285, 176, 286],
        [  0,   0,   0,  ..., 176, 286, 287],
        [  0,   0,   0,  ..., 286, 287, 288]], device='cuda:0')

In [145]:
padded_training_sequence

tensor([[  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        [  0,   0,   0,  ...,   0,   4,   5],
        ...,
        [  0,   0,   0,  ..., 285, 176, 286],
        [  0,   0,   0,  ..., 176, 286, 287],
        [  0,   0,   0,  ..., 286, 287, 288]])

In [146]:
x=padded_training_sequence[:,:-1]
y=padded_training_sequence[:,-1]

In [147]:
class CustomDataset(Dataset):

  def __init__(self, x, y):
    self.x = x
    self.y = y

  def __len__(self):
    return self.x.shape[0]

  def __getitem__(self, idx):
    return self.x[idx], self.y[idx]

In [148]:
dataset=CustomDataset(x,y)

In [149]:
dataloader=DataLoader(dataset, batch_size=32, shuffle=True, pin_memory=True)

In [150]:
for input,output in dataloader:
  print(input,output)

tensor([[  0,   0,   0,  ...,   0,  55,   8],
        [  0,   0,   0,  ..., 267, 252, 268],
        [  0,   0,   0,  ...,  95,  80,  96],
        ...,
        [  0,   0,   0,  ...,   0,   0, 225],
        [  0,   0,   0,  ...,  43,  68,  69],
        [  0,   0,   0,  ..., 163, 164, 109]]) tensor([  9,  30,  78,  24, 105,  23,  75,  33, 253,  81,   2,  65, 149,   2,
         37,  78,  50, 111,  27, 166, 100, 141, 212,  22,  59, 195, 242,  19,
          6, 132,  70, 209])
tensor([[  0,   0,   0,  ...,  82, 156,  23],
        [  0,   0,   0,  ...,  85,  86, 136],
        [  0,   0,   0,  ...,   0,   0,  44],
        ...,
        [  0,   0,   0,  ..., 146,  24,  25],
        [  0,   0,   0,  ...,   0,   0, 123],
        [  0,   0,   0,  ..., 131,  89, 132]]) tensor([ 24, 127,  45, 127,   2,  99, 165, 146,  82, 165,  28, 254, 287,  64,
          8, 176,  38,  45, 233,  23,  30, 176, 131,  75,   0, 125,  46,  76,
        132,   2,  45,  22])
tensor([[  0,   0,   0,  ..., 191,   2, 183],
    

In [151]:
from torch._prims_common import Dim
class LSTMModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,100) #dimention=100
    self.lstm=nn.LSTM(100,150,batch_first=True) #Number of nuerons in the internal hidden nn=150
    self.fc=nn.Linear(150,vocab_size)

  def forward(self,x):
    embedded=self.embedding(x)
    intermediate_hidden_state,(h_t,c_t)=self.lstm(embedded)
    output=self.fc(h_t.squeeze(0))
    return output

For Understanding

In [152]:
x=nn.Embedding(289,100)
y=nn.LSTM(100,150,batch_first=True)
z=nn.Linear(150,289)

In [153]:
a=dataset[0][0].unsqueeze(0)

In [154]:
b=x(a)

In [155]:
c,d=y(b)

In [156]:
c.shape
# set of all intermediate hidden states

torch.Size([1, 61, 150])

In [157]:
e,f=d

In [158]:
e.shape
# set of all h_t

torch.Size([1, 1, 150])

In [159]:
f.shape
#set of all c_t

torch.Size([1, 1, 150])

In [160]:
z(f.squeeze(0)).shape

torch.Size([1, 289])

Model and Training Loop


In [161]:
model = LSTMModel(len(vocab))
model.to(device)

LSTMModel(
  (embedding): Embedding(289, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=289, bias=True)
)

In [162]:
epochs=50
learning_rate=0.001

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

In [163]:
#training loop
for epoch in range(epochs):
  total_epoch_loss=0
  for batch_x, batch_y in dataloader:

    batch_x=batch_x.to(device)
    batch_y=batch_y.to(device)

    #zero gradient
    optimizer.zero_grad()

    #forward pass
    y_pred=model(batch_x)

    #loss
    loss = criterion(y_pred,batch_y)

    #backward pass
    loss.backward()

    #update params
    optimizer.step()

    #print loss
    total_epoch_loss+=loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_epoch_loss:.4f}")

Epoch: 1, Loss: 166.6554
Epoch: 2, Loss: 146.7097
Epoch: 3, Loss: 133.9360
Epoch: 4, Loss: 120.7895
Epoch: 5, Loss: 109.0534
Epoch: 6, Loss: 96.8392
Epoch: 7, Loss: 86.6593
Epoch: 8, Loss: 77.2643
Epoch: 9, Loss: 68.3576
Epoch: 10, Loss: 60.1365
Epoch: 11, Loss: 52.6086
Epoch: 12, Loss: 46.6395
Epoch: 13, Loss: 40.8370
Epoch: 14, Loss: 35.8919
Epoch: 15, Loss: 31.5389
Epoch: 16, Loss: 27.3750
Epoch: 17, Loss: 24.3050
Epoch: 18, Loss: 21.3857
Epoch: 19, Loss: 18.9979
Epoch: 20, Loss: 17.0056
Epoch: 21, Loss: 15.3812
Epoch: 22, Loss: 13.8623
Epoch: 23, Loss: 12.6316
Epoch: 24, Loss: 11.4880
Epoch: 25, Loss: 10.6920
Epoch: 26, Loss: 10.0121
Epoch: 27, Loss: 9.2366
Epoch: 28, Loss: 8.5764
Epoch: 29, Loss: 8.1788
Epoch: 30, Loss: 7.6799
Epoch: 31, Loss: 7.2331
Epoch: 32, Loss: 6.9933
Epoch: 33, Loss: 6.6397
Epoch: 34, Loss: 6.4125
Epoch: 35, Loss: 6.0315
Epoch: 36, Loss: 5.9663
Epoch: 37, Loss: 5.7450
Epoch: 38, Loss: 5.7254
Epoch: 39, Loss: 5.4435
Epoch: 40, Loss: 5.3245
Epoch: 41, Loss: 5

In [166]:
# prediction

def prediction(model, vocab, text):

  # tokenize
  tokenized_text = word_tokenize(text.lower())

  # text -> numerical indices
  numerical_text = text_to_indices(tokenized_text, vocab)

  # padding
  padded_text = torch.tensor([0] * (61 - len(numerical_text)) + numerical_text, dtype=torch.long).unsqueeze(0)

  padded_text = padded_text.to(device)

  # send to model
  output = model(padded_text)

  # predicted index
  value, index = torch.max(output, dim=1)

  # merge with text
  return text + " " + list(vocab.keys())[index]


In [167]:
prediction(model, vocab, "The course follows a monthly")

'The course follows a monthly subscription'

In [169]:
import time

num_tokens = 10
input_text = "You can check the"

for i in range(num_tokens):
  output_text = prediction(model, vocab, input_text)
  print(output_text)
  input_text = output_text
  time.sleep(0.5)

You can check the detailed
You can check the detailed syllabus
You can check the detailed syllabus here
You can check the detailed syllabus here -
You can check the detailed syllabus here - https
You can check the detailed syllabus here - https :
You can check the detailed syllabus here - https : //learnwith.campusx.in/courses/campusx-data-science-mentorship-program-637339afe4b0615a1bbed390
You can check the detailed syllabus here - https : //learnwith.campusx.in/courses/campusx-data-science-mentorship-program-637339afe4b0615a1bbed390 //learnwith.campusx.in/courses/campusx-data-science-mentorship-program-637339afe4b0615a1bbed390
You can check the detailed syllabus here - https : //learnwith.campusx.in/courses/campusx-data-science-mentorship-program-637339afe4b0615a1bbed390 //learnwith.campusx.in/courses/campusx-data-science-mentorship-program-637339afe4b0615a1bbed390 //learnwith.campusx.in/courses/campusx-data-science-mentorship-program-637339afe4b0615a1bbed390
You can check the detail

In [170]:
dataloader1 = DataLoader(dataset, batch_size=32, shuffle=False)

In [171]:
# Function to calculate accuracy
def calculate_accuracy(model, dataloader, device):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():  # No need to compute gradients
        for batch_x, batch_y in dataloader1:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            # Get model predictions
            outputs = model(batch_x)

            # Get the predicted word indices
            _, predicted = torch.max(outputs, dim=1)

            # Compare with actual labels
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    accuracy = correct / total * 100
    return accuracy

# Compute accuracy
accuracy = calculate_accuracy(model, dataloader, device)
print(f"Model Accuracy: {accuracy:.2f}%")


Model Accuracy: 95.65%
